# GWAS Summary Statistics Harmonization using Nextflow

This notebook demonstrates how to run the [EBISPOT GWAS Sumstats Harmoniser](https://github.com/EBISPOT/gwas-sumstats-harmoniser) pipeline using Nextflow and Docker with local sample data.

### Fixes & Improvements:
-   **Memory Optimization**: Creates a custom config to limit memory usage (preventing crashes on local machines).
-   **Column Standardization**: Renames input columns to match the pipeline's expected format.
-   **Robust Data Prep**: Copies files instead of symlinking to ensure Docker container access.
-   **Reference Naming**: Renames reference files to `homo_sapiens-chr{N}.vcf.gz` as required by the pipeline.
-   **Valid Parquet**: Generates a dummy Parquet file with the correct schema (`CHR`, `POS`, `ID`) to satisfy internal pipeline checks.
-   **Pipeline Parameters**: Uses `--chromlist` and `--to_build 37` tailored for the sample data.

## prerequisites
1.  **Nextflow**: Must be installed and available in the system path.
2.  **Docker**: Must be running.
3.  **Sample Data**: Located in `../gwaslab-sample-data`.
4.  **Python Libraries**: `pandas` (for data prep).

In [ ]:
# Cell 1: Environment Check
import subprocess
import os
import pandas as pd
import sys

def check_tool(command):
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print(f"{command.split()[0]} is available.")
    except subprocess.CalledProcessError:
        print(f"{command.split()[0]} is NOT available or failed to run.")
        raise

print("Checking environment...")
try:
    check_tool("nextflow -version")
    check_tool("docker --version")
    import pandas
    print("pandas is available")
except Exception as e:
    print(f"Environment error: {e}")

Checking environment...
✅ nextflow is available.
✅ docker is available.
✅ pandas is available


In [ ]:
# Cell 2: Define Data Paths & Data Preparation
import os
import shutil

# Absolute paths
base_dir = os.path.abspath("../gwaslab-sample-data")
notebook_dir = os.getcwd()
workspace_dir = os.path.join(notebook_dir, "run_workspace")

# Clean previous workspace to ensure fresh state
if os.path.exists(workspace_dir):
    shutil.rmtree(workspace_dir)
os.makedirs(workspace_dir, exist_ok=True)

# Input GWAS Summary Statistics
input_filename = "bbj_t2d_hm3_chr7_variants.txt.gz"
source_input = os.path.join(base_dir, input_filename)
target_input_name = "standardized_input.tsv"
target_input = os.path.join(workspace_dir, target_input_name)

print("Processing Input File...")
# Read and Standardize Header (CRITICAL STEP)
# The pipeline expects specific column names (defined in common_constants.py)
if os.path.exists(source_input):
    df = pd.read_csv(source_input, sep='\t')
    rename_map = {
        'CHR': 'chromosome',
        'POS': 'base_pair_location',
        'EA': 'effect_allele',
        'NEA': 'other_allele',
        'EAF': 'effect_allele_frequency',
        'BETA': 'beta',
        'SE': 'standard_error',
        'P': 'p_value',
        'rsID': 'rsid',
        'SNPID': 'variant_id'
    }
    print(f"  Columns before: {list(df.columns)}")
    df.rename(columns=rename_map, inplace=True)
    print(f"  Columns after: {list(df.columns)}")
    
    # Save as TSV (Docker friendly location)
    df.to_csv(target_input, sep='\t', index=False)
    print(f"Input processed and saved: {target_input}")
else:
    print(f"Input missing: {source_input}")

# Prepare References Directory
ref_dir_workspace = os.path.join(workspace_dir, "references")
os.makedirs(ref_dir_workspace, exist_ok=True)

# Source References from gwaslab-sample-data
src_vcf = os.path.join(base_dir, "1kg_eas_hg19.chr7_126253550_128253550.vcf.gz")

# Destination Paths (Renaming to match pipeline 'homo_sapiens-chrN' convention)
dst_vcf = os.path.join(ref_dir_workspace, "homo_sapiens-chr7.vcf.gz")
dst_tbi = os.path.join(ref_dir_workspace, "homo_sapiens-chr7.vcf.gz.tbi")
dst_parquet = os.path.join(ref_dir_workspace, "homo_sapiens-chr7.parquet")

# Copy VCF/TBI (Copying avoids Docker symlink volume issues)
if os.path.exists(src_vcf):
    shutil.copy(src_vcf, dst_vcf)
    if os.path.exists(src_vcf + ".tbi"):
        shutil.copy(src_vcf + ".tbi", dst_tbi)
        print(f"Copied Reference VCF/TBI")
    else:
        print("Missing TBI file")
else:
    print(f"Reference missing: {src_vcf}")

# Create Dummy Parquet (Pipeline Requirement: Must have specific schema)
if not os.path.exists(dst_parquet):
    # Schema required by map_to_build_nf.py
    df_parquet = pd.DataFrame({
        'CHR': ['7'], 
        'POS': [1], 
        'ID': ['rsDummy'],
        'REF': ['A'],
        'ALT': ['T']
    })
    try:
        df_parquet.to_parquet(dst_parquet)
        print(f"Created dummy parquet: {os.path.basename(dst_parquet)}")
    except Exception as e:
        print(f"Failed to create parquet: {e}")

Processing Input File...


/tmp/ipykernel_13034/2694446868.py:25: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(source_input, sep='\t')


  Columns before: ['SNPID', 'CHR', 'POS', 'EA', 'NEA', 'EAF', 'BETA', 'SE', 'P', 'N', 'DIRECTION', 'STATUS', 'rsID']
  Columns after: ['variant_id', 'chromosome', 'base_pair_location', 'effect_allele', 'other_allele', 'effect_allele_frequency', 'beta', 'standard_error', 'p_value', 'N', 'DIRECTION', 'STATUS', 'rsid']
✅ Input processed and saved: /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/standardized_input.tsv
✅ Copied Reference VCF/TBI
✅ Created dummy parquet: homo_sapiens-chr7.parquet


In [ ]:
# Cell 3: Create Metadata YAML and Custom Config

yaml_content = f"""
# Study meta-data
date_metadata_last_modified: 2023-01-01
# Genotyping Information
genome_assembly: GRCh37
coordinate_system: 1-based
# Summary Statistic information
data_file_name: {target_input_name}
file_type: GWAS-SSF v0.1
data_file_md5sum: 00000000000000000000000000000000
# Harmonization status
is_harmonised: false
is_sorted: false
"""

yaml_path = target_input + "-meta.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"Created Metadata: {yaml_path}")

# Custom Config to Reduce Memory Usage (Fixes 'Process requirement exceeds available memory')
# We limit all potential heavy processes to 4GB to be safe on local machine
config_content = """
process {
    withName: 'map_to_build' { memory = '4 GB' }
    withName: 'ten_percent_counts' { memory = '4 GB' }
    withName: 'generate_strand_counts' { memory = '4 GB' }
    withName: 'summarise_strand_counts' { memory = '4 GB' }
    withName: 'harmonization' { memory = '4 GB' }
    withName: 'qc' { memory = '4 GB' }
}
"""
config_path = os.path.join(workspace_dir, "memory_fix.config")
with open(config_path, "w") as f:
    f.write(config_content)
print(f"Created Config: {config_path}")

✅ Created Metadata: /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/standardized_input.tsv-meta.yaml
✅ Created Config: /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/memory_fix.config


## Run Harmonization Pipeline

We use `nextflow run` with the following adjustments:
- `-c .../memory_fix.config`: Applies our custom memory limits.
- `--chromlist 7,`: Specifies the chromosome list (comma is needed to force string interpretation).
- `--to_build 37`: Sets target build to 37 (matching input) to bypass Liftover if not needed.
- `--file`: Points to the **standardized** input file.
- `--ref`: Points to the directory where we copied and renamed references.

In [ ]:
# Execute Pipeline

# Construct the command
cmd = f"""
nextflow run EBISPOT/gwas-sumstats-harmoniser \\
    -r v1.1.10 \\
    -profile docker \\
    -c {os.path.abspath(os.path.join(workspace_dir, 'memory_fix.config'))} \\
    --harm \\
    --file {target_input} \\
    --ref {ref_dir_workspace} \\
    --chromlist 7, \\
    --to_build 37 \\
    --threshold 0.99
"""

print("Executing command:")
print(cmd)

!{cmd}

Executing command:

nextflow run EBISPOT/gwas-sumstats-harmoniser \
    -r v1.1.10 \
    -profile docker \
    -c /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/memory_fix.config \
    --harm \
    --file /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/standardized_input.tsv \
    --ref /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/references \
    --chromlist 7, \
    --to_build 37 \
    --threshold 0.99

Nextflow 25.10.3 is available - Please consider updating your version to it

 N E X T F L O W   ~  version 25.10.2

Launching `https://github.com/EBISPOT/gwas-sumstats-harmoniser` [condescending_ride] DSL2 - revision: 436c17a91c [v1.1.10]

WARN: Config setting `docker.userEmulation` is not supported anymore
Start harmonising files
Harmonizing the file /home/dawit/Documents/Projects/rejuve-bio2/GalaxyTools/use-gwaslab/Notebooks/run_workspace/standardized_